In [1]:
import os
import numpy as np
import pandas as pd
import librosa
import natsort
from sklearn.preprocessing import StandardScaler

# 경로 설정
train_path = '/Users/withmocha/Desktop/DATA/SW DACON(24)/Data/row/train/'
test_path = '/Users/withmocha/Desktop/DATA/SW DACON(24)/Data/row/test/'

train_list = natsort.natsorted(os.listdir(train_path))
test_list = natsort.natsorted(os.listdir(test_path))

In [2]:
# train.csv 파일 읽기
train_metadata = pd.read_csv("/Users/withmocha/Desktop/DATA/SW DACON(24)/Data/row/train.csv")
train_metadata = train_metadata.sort_values(['id', 'path', 'label']).reset_index(drop=True)
train_metadata

,id,path,label
0,AAACWKPZ,./train/AAACWKPZ.ogg,fake
1,AAAQOZYI,./train/AAAQOZYI.ogg,fake
2,AAAWBXQE,./train/AAAWBXQE.ogg,real
3,AABHDRLX,./train/AABHDRLX.ogg,real
4,AABXXHMU,./train/AABXXHMU.ogg,real
...,...,...,...
55433,ZZZMJSBX,./train/ZZZMJSBX.ogg,fake
55434,ZZZRTSKN,./train/ZZZRTSKN.ogg,fake
55435,ZZZSAANM,./train/ZZZSAANM.ogg,real
55436,ZZZSYEYS,./train/ZZZSYEYS.ogg,fake


In [3]:
train_list

['AAACWKPZ.ogg',
 'AAAQOZYI.ogg',
 'AAAWBXQE.ogg',
 'AABHDRLX.ogg',
 'AABXXHMU.ogg',
 'AACAYDKC.ogg',
 'AACBAWMZ.ogg',
 'AACEAXVU.ogg',
 'AACHVERO.ogg',
 'AACYAWEH.ogg',
 'AADDNUMU.ogg',
 'AADKWHEE.ogg',
 'AADPDAER.ogg',
 'AADQUEFI.ogg',
 'AAECEROJ.ogg',
 'AAEYTHSY.ogg',
 'AAFFVAWD.ogg',
 'AAFNYPXE.ogg',
 'AAGDMPXZ.ogg',
 'AAGIITHH.ogg',
 'AAGXILIN.ogg',
 'AAGZDQDL.ogg',
 'AAHKQWBB.ogg',
 'AAIGXLUD.ogg',
 'AAIINGWH.ogg',
 'AAIKZLBQ.ogg',
 'AAIXOVVV.ogg',
 'AAIYWHUF.ogg',
 'AAJEQLVT.ogg',
 'AAJKXPIW.ogg',
 'AAJMPRLS.ogg',
 'AAJSFGQB.ogg',
 'AAKKEAOL.ogg',
 'AAKXVWXI.ogg',
 'AALITDLH.ogg',
 'AALMMBOZ.ogg',
 'AALVLTSV.ogg',
 'AAMMREFE.ogg',
 'AANJJHYA.ogg',
 'AANOXTME.ogg',
 'AANUFJBH.ogg',
 'AAOEAZJW.ogg',
 'AAOFKKOG.ogg',
 'AAOYUWCN.ogg',
 'AAPELQFY.ogg',
 'AAPGWSKV.ogg',
 'AAPWJSUB.ogg',
 'AAQBLCRK.ogg',
 'AAQBNHMD.ogg',
 'AAQBQXCT.ogg',
 'AAQFLJFN.ogg',
 'AAQJRBNN.ogg',
 'AAQNTEKV.ogg',
 'AAQQXFSY.ogg',
 'AARBSIOH.ogg',
 'AARGVJZI.ogg',
 'AARGWRKZ.ogg',
 'AARRJAAQ.ogg',
 'AASWUMSW.ogg

In [4]:
Y=pd.DataFrame(train_metadata['label'],columns=['label'])
Y

,label
0,fake
1,fake
2,real
3,real
4,real
...,...
55433,fake
55434,fake
55435,real
55436,fake


In [5]:
# 데이터 증강 함수
import random
def augment_audio(y):
    # 시간 축소
    if random.random() < 0.5:
        y = librosa.effects.time_stretch(y, rate=random.uniform(0.8, 1.2))
    # 피치 변경
    if random.random() < 0.5:
        y = librosa.effects.pitch_shift(y, sr=sample_rate, n_steps=random.randint(-3, 3))
    return y

In [6]:
# Constants
duration = 2  # seconds
sample_rate = 32000  # Hz
n_mels = 128
n_mfcc = 40


In [7]:
# Initialize lists for features and labels
train_feature = []

for i in range(len(train_list)):
    x, _ = librosa.load(train_path + train_list[i], sr=sample_rate, duration=duration)
    
    # 데이터 증강
    x = augment_audio(x)
    
    # 노이즈 감소
    x = librosa.effects.preemphasis(x)
    
    # Extract log-mel spectrogram
    S = librosa.feature.melspectrogram(y=x, sr=sample_rate, n_mels=n_mels)
    log_S = librosa.power_to_db(S, ref=np.max)
    
    # Extract MFCC
    mfcc = librosa.feature.mfcc(S=log_S, n_mfcc=n_mfcc)
    
    # Compute delta and delta-delta MFCC
    width = min(7, mfcc.shape[1]) if mfcc.shape[1] >= 3 else 3
    delta_mfcc = librosa.feature.delta(mfcc, width=width)
    delta2_mfcc = librosa.feature.delta(mfcc, order=2, width=width)
    
    # Extract spectral contrast
    spectral_contrast = librosa.feature.spectral_contrast(S=log_S, sr=sample_rate)
    
    # Concatenate features along the MFCC axis
    combined_features = np.concatenate((mfcc, delta_mfcc, delta2_mfcc, spectral_contrast), axis=0)
    
    # Normalize the combined features
    scaler = StandardScaler()
    normalized_features = scaler.fit_transform(combined_features)
    
    # Append to the feature list
    train_feature.append(normalized_features.flatten())



In [8]:
train_feature_df = pd.DataFrame(train_feature)
train_feature_df

,0,1,2,3,4,5,6,7,8,9,...,19929,19930,19931,19932,19933,19934,19935,19936,19937,19938
0,-10.680473,-10.065905,-8.485398,-8.471478,-8.938723,-9.573242,-9.778496,-9.528599,-9.170987,-8.849465,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,-11.071263,-11.079940,-10.551203,-10.089887,-9.634271,-9.297155,-9.277541,-9.482996,-9.715906,-10.094424,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-11.061636,-10.900703,-9.897691,-9.627090,-9.460627,-9.390776,-9.529352,-9.783495,-10.150701,-10.327344,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,-10.501434,-9.739539,-8.956120,-9.592299,-10.143098,-10.398886,-10.340263,-10.351754,-10.469296,-10.385562,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,-11.117754,-11.013177,-10.541680,-10.596990,-10.668945,-10.714156,-10.677109,-10.532436,-10.372413,-10.261147,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55433,-11.144724,-11.137963,-10.819232,-10.458742,-10.402115,-10.462618,-10.440723,-10.395901,-10.475930,-10.594594,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
55434,-10.425381,-9.859460,-9.212831,-9.264336,-9.338524,-9.558168,-9.837779,-10.059920,-10.196332,-10.262708,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
55435,-8.136928,-8.514102,-8.929909,-9.356843,-9.693064,-10.213511,-10.750908,-10.973743,-11.060165,-11.075549,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
55436,-10.992670,-10.891329,-10.557309,-10.454976,-10.429793,-10.488954,-10.582969,-10.744830,-10.849541,-10.596731,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
train_feature_df2=train_feature_df.dropna(axis=1)

In [10]:
train_feature_df2

,0,1,2,3,4,5,6,7,8,9,...,1006,1007,1008,1009,1010,1011,1012,1013,1014,1015
0,-10.680473,-10.065905,-8.485398,-8.471478,-8.938723,-9.573242,-9.778496,-9.528599,-9.170987,-8.849465,...,-0.453353,-0.322508,-0.087017,0.304063,0.404823,0.479159,0.535449,0.511438,0.202025,0.074921
1,-11.071263,-11.079940,-10.551203,-10.089887,-9.634271,-9.297155,-9.277541,-9.482996,-9.715906,-10.094424,...,-0.193274,-0.286983,0.044365,0.068283,0.268024,0.378609,0.477951,0.543279,0.465428,0.377558
2,-11.061636,-10.900703,-9.897691,-9.627090,-9.460627,-9.390776,-9.529352,-9.783495,-10.150701,-10.327344,...,0.133401,0.084898,0.158464,-0.037385,-0.031830,-0.065905,-0.144708,-0.238556,-0.262587,-0.109534
3,-10.501434,-9.739539,-8.956120,-9.592299,-10.143098,-10.398886,-10.340263,-10.351754,-10.469296,-10.385562,...,-0.547253,-0.535377,0.416520,0.387058,0.726899,0.558501,0.497426,0.449314,0.332968,0.456198
4,-11.117754,-11.013177,-10.541680,-10.596990,-10.668945,-10.714156,-10.677109,-10.532436,-10.372413,-10.261147,...,0.029022,-0.007236,0.113250,0.147934,0.123936,0.134960,0.118806,0.215570,0.201303,0.227695
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55433,-11.144724,-11.137963,-10.819232,-10.458742,-10.402115,-10.462618,-10.440723,-10.395901,-10.475930,-10.594594,...,-0.546889,-0.305863,0.020189,0.080815,0.032809,-0.131100,-0.412744,-0.478068,-0.421102,-0.405390
55434,-10.425381,-9.859460,-9.212831,-9.264336,-9.338524,-9.558168,-9.837779,-10.059920,-10.196332,-10.262708,...,-0.281078,-0.343358,-0.430669,-0.537894,-0.542315,-0.594209,-0.467926,-0.462328,-0.471181,-0.395152
55435,-8.136928,-8.514102,-8.929909,-9.356843,-9.693064,-10.213511,-10.750908,-10.973743,-11.060165,-11.075549,...,0.005537,0.033486,0.628226,0.426825,0.151324,0.045645,0.201020,0.437524,0.442280,0.383002
55436,-10.992670,-10.891329,-10.557309,-10.454976,-10.429793,-10.488954,-10.582969,-10.744830,-10.849541,-10.596731,...,-0.634200,-0.632286,0.220390,0.238211,0.220074,0.083456,0.072693,0.168384,0.335541,0.345256


In [11]:
save_data=pd.concat([train_feature_df2,Y],axis=1)

In [12]:
save_data.to_csv('train data columns.csv')

In [20]:
train_feature_df3 = train_feature_df

In [21]:
train_feature_df3

,0,1,2,3,4,5,6,7,8,9,...,19929,19930,19931,19932,19933,19934,19935,19936,19937,19938
0,-10.680473,-10.065905,-8.485398,-8.471478,-8.938723,-9.573242,-9.778496,-9.528599,-9.170987,-8.849465,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,-11.071263,-11.079940,-10.551203,-10.089887,-9.634271,-9.297155,-9.277541,-9.482996,-9.715906,-10.094424,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-11.061636,-10.900703,-9.897691,-9.627090,-9.460627,-9.390776,-9.529352,-9.783495,-10.150701,-10.327344,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,-10.501434,-9.739539,-8.956120,-9.592299,-10.143098,-10.398886,-10.340263,-10.351754,-10.469296,-10.385562,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,-11.117754,-11.013177,-10.541680,-10.596990,-10.668945,-10.714156,-10.677109,-10.532436,-10.372413,-10.261147,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55433,-11.144724,-11.137963,-10.819232,-10.458742,-10.402115,-10.462618,-10.440723,-10.395901,-10.475930,-10.594594,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
55434,-10.425381,-9.859460,-9.212831,-9.264336,-9.338524,-9.558168,-9.837779,-10.059920,-10.196332,-10.262708,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
55435,-8.136928,-8.514102,-8.929909,-9.356843,-9.693064,-10.213511,-10.750908,-10.973743,-11.060165,-11.075549,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
55436,-10.992670,-10.891329,-10.557309,-10.454976,-10.429793,-10.488954,-10.582969,-10.744830,-10.849541,-10.596731,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
train_feature_df4=train_feature_df3.dropna(axis=0)
train_feature_df4

,0,1,2,3,4,5,6,7,8,9,...,19929,19930,19931,19932,19933,19934,19935,19936,19937,19938
226,-10.332452,-10.653242,-9.921116,-9.828106,-9.909056,-10.179271,-10.351407,-10.517357,-10.501415,-10.425664,...,0.071811,0.073234,0.068456,0.066757,0.070969,0.073286,0.073101,0.079294,0.092678,0.086852
3896,-11.110371,-11.026034,-10.508208,-10.575073,-10.700702,-10.766001,-10.696896,-10.535518,-10.386226,-10.288738,...,0.125204,0.130181,0.127780,0.127217,0.128752,0.127653,0.124642,0.128464,0.125920,0.113264
4748,-10.821043,-10.566454,-10.165197,-10.280624,-10.369719,-10.418716,-10.503160,-10.598422,-10.674245,-10.770985,...,0.108168,0.103385,0.094516,0.091256,0.094867,0.091023,0.093556,0.091300,0.090713,0.096513
5205,-10.937991,-10.942903,-10.459420,-10.266849,-10.089887,-9.940216,-9.793228,-9.717905,-9.633093,-9.585300,...,0.104985,0.105733,0.103345,0.104898,0.106309,0.105874,0.106072,0.101938,0.103386,0.103192
8412,-10.911749,-10.654518,-10.169845,-10.267769,-10.364705,-10.425001,-10.505059,-10.574101,-10.564409,-10.543023,...,0.100516,0.103720,0.105002,0.108179,0.107293,0.100171,0.103575,0.112232,0.087287,0.079061
8501,-11.068932,-11.016607,-10.558576,-10.482119,-10.261645,-10.090131,-9.986119,-9.965896,-9.997606,-10.035193,...,0.119085,0.119479,0.117914,0.116776,0.111196,0.107371,0.105211,0.087236,0.084227,0.079126
8723,-10.785468,-10.531050,-10.246325,-10.365131,-10.395986,-10.402717,-10.433034,-10.429281,-10.465834,-10.520872,...,0.089628,0.093175,0.104663,0.100481,0.098536,0.094077,0.091578,0.091247,0.082855,0.074746
10677,-11.076220,-11.035813,-10.897844,-10.868905,-10.781412,-10.663881,-10.581081,-10.472749,-10.354945,-10.221459,...,0.110432,0.115605,0.119441,0.117407,0.117085,0.107981,0.100465,0.099540,0.088859,0.079760
10748,-10.983168,-10.905937,-10.345457,-10.196399,-10.130197,-10.068248,-10.041068,-9.965592,-9.842871,-9.805907,...,0.090611,0.093708,0.101390,0.100699,0.099734,0.095574,0.096933,0.102854,0.105754,0.096783
15196,-11.130369,-11.060469,-10.563182,-10.410269,-10.227773,-10.128735,-10.082428,-9.962530,-9.958223,-9.975958,...,0.100498,0.101515,0.094142,0.091790,0.087476,0.088997,0.092065,0.092320,0.094995,0.090979


In [16]:
train_feature_df3.to_csv("train data row.csv")